# DAT540: Generative AI

* Lectures: Vinay Setty, Petra Galuscakova.
* Teaching Assistant: Gabriel Iturra-Bocaz.

## How to use Ollama UiS?

But first...

### What is Ollama?

[Ollama](https://ollama.com/) is a tool that allows you to run Large Language Models (LLMs) locally on your own computer or server, without relying on external cloud-based APIs.

With Ollama, you can:

* Download models such as Llama, Mistral, Qwen, DeepSeek, and others-

* Run them locally through a command-line interface or API.

* Integrate them easily into Python applications.

* Maintain full control over your data (privacy).

* Reduce costs by avoiding per-token cloud API fees.

Technically, Ollama acts as a local LLM server, exposing an HTTP API that enables text generation, chat interactions, and embeddings.

### How to use Ollama at UiS

You can find all the relevant information here: https://ollama.ux.uis.no/

First you need to import it your OLLAMA_API_KEY

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

Import the Ollama library, especially the Client component

In [3]:
import os
from ollama import Client

Define the client and set the host and headers, including the OLLAMA_API_KEY.

In [4]:
client = Client(
    host="https://ollama.ux.uis.no",
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_API_KEY')}",
        "Content-Type": "application/json"
    }
)

Make a call the Ollama API

In [6]:
resp = client.generate(
    model="qwen3:0.6b",
    prompt="What is the capital of Norway?",
    think=False,
    stream=False,
)
resp.response

'The capital of Norway is **Oslo**.'

#### Nice, but... how do I get my token API Key?

* Log in to Open WebUI at https://openwebui.ux.uis.no using your UiS credentials.

* Navigate to Settings → Account.

* Generate a new API key.

* Copy and store your OLLAMA_API_KEY securely.

* Never commit your API key to version control (e.g., GitHub).

* Then you can save in ```.env``` file and call it using ```os``` library in Python.

#### IMDB Dataset of 50K Movie Reviews

IMDB dataset having 50K movie reviews for natural language processing or Text analytics.
This is a dataset for binary sentiment classification containing substantially more data than previous benchmark datasets. We provide a set of 25,000 highly polar movie reviews for training and 25,000 for testing. So, predict the number of positive and negative reviews using either classification or deep learning algorithms.
For more dataset information, please go through the following link,
http://ai.stanford.edu/~amaas/data/sentiment/

In [9]:
import pandas as pd
from pprint import pprint

In [10]:
CSV_URL = "https://raw.githubusercontent.com/windi-wulandari/sentiment-analysis-IMDB/refs/heads/main/dataset/IMDb_Reviews.csv"
df = pd.read_csv(CSV_URL)

df.head()

,review,sentiment
0,My family and I normally do not watch local mo...,1
1,"Believe it or not, this was at one time the wo...",0
2,"After some internet surfing, I found the ""Home...",0
3,One of the most unheralded great works of anim...,1
4,"It was the Sixties, and anyone with long hair ...",0


Dataset shape:

In [25]:
df.shape

(50000, 2)

Column names and types:

In [26]:
df.dtypes

review       object
sentiment     int64
dtype: object

Statistics

In [27]:
df.describe()

,sentiment
count,50000.000000
mean,0.500000
std,0.500005
min,0.000000
25%,0.000000
50%,0.500000
75%,1.000000
max,1.000000


#### What is Prompt Engineering?

Prompt engineering is the skill of writing effective instructions for AI models (like ChatGPT) so they produce accurate, useful, and relevant responses.

Since AI models respond based on patterns in the input, how you ask matters. A vague prompt gives vague output. A clear, structured prompt gives focused results.

Good prompt engineering usually involves:

* Being specific about the task

* Providing context

* Defining the format you want

* Giving examples when needed

* Setting constraints (length, tone, structure)

It’s not about tricking the AI — it’s about guiding it.

##### Zero-Shot Prompting


Zero-shot means you give the AI a task without any examples.

You simply describe what you want.

Example:

> "Translate this sentence into Spanish: I love learning."

The model performs the task without seeing examples first.

When to use it:

* The task is simple or common

* You want fast responses

* You don’t need a very specific format

In [7]:
prompt = """You are a sentiment classifier.

Task:
Classify the sentiment of the review as exactly one of:
POSITIVE, NEGATIVE

Rules:
- Output ONLY the label (POSITIVE or NEGATIVE).
- Do not add explanations.

Review:
{text}

Label:"""

In [11]:
sample = df.sample(1)
review = sample["review"].iloc[0]
label = "POSITIVE" if sample["sentiment"].iloc[0] == 1 else "NEGATIVE"

In [12]:
formatted_prompt = prompt.format(text=review)
pprint(formatted_prompt)

('You are a sentiment classifier.\n'
 '\n'
 'Task:\n'
 'Classify the sentiment of the review as exactly one of:\n'
 'POSITIVE, NEGATIVE\n'
 '\n'
 'Rules:\n'
 '- Output ONLY the label (POSITIVE or NEGATIVE).\n'
 '- Do not add explanations.\n'
 '\n'
 'Review:\n'
 'Really good horror flick featuring to of the greatest, Boris Karloff and '
 'Bela Lugosi. Dr. Janos Rukh(Karloff)is on an expedition in Africa trying to '
 'find an ancient meteorite. After finding it, Rukh is poisoned by the its '
 'radiation. All he touches dies and the dark side of Rukh makes him become an '
 'egotistic murderer. His friend, Dr. Felix Benet(Lugosi)finds a limited '
 'remedy to the problem and at the same time realizes the radiation could be '
 'used for the good of mankind by curing diseases. The two fiends will battle '
 'over the radiations possibilities. Pretty good special effects. Others in '
 'the cast: Frances Drake, Frank Lawton, Beulah Bondi and Frank Reicher.\n'
 '\n'
 'Label:')


In [18]:
client.generate(
    model="qwen3:0.6b",
    prompt=formatted_prompt,
    think=False,
    stream=False,
).response

'NEGATIVE'

In [19]:
label

'POSITIVE'

##### Few-Shot Prompting

Few-shot means you provide a small number of examples before asking the AI to continue or perform the task.

You show the pattern first.

```
Review: Happy → Label: Positive
Input: Sad → Output: Negative
Input: Angry → Output:
```

Now the model understands the pattern and continues accordingly.

When to use it:

* You need a specific format

* The task is unusual

* Consistency matters

* You want higher accuracy

In [20]:
def find_columns(df: pd.DataFrame):
    cols = {c.lower(): c for c in df.columns}
    text_col = cols.get("review") or cols.get("text") or cols.get("sentence")
    label_col = cols.get("sentiment") or cols.get("label") or cols.get("class")
    if not text_col or not label_col:
        raise ValueError(f"Could not infer text/label columns. Columns: {list(df.columns)}")
    return text_col, label_col

def normalize_label(x: str) -> str:
    s = str(x).strip().lower()
    if s in {"positive", "pos", "1", "yes"}:
        return "POSITIVE"
    if s in {"negative", "neg", "0", "no"}:
        return "NEGATIVE"
    return str(x).strip().upper()

def select_five_examples(
    df: pd.DataFrame,
    text_col: str,
    label_col: str
):

    tmp = df[[text_col, label_col]].dropna().copy()
    tmp[label_col] = tmp[label_col].apply(normalize_label)

    vc = tmp[label_col].value_counts()

    if len(vc) >= 2:
        maj = vc.index[0]
        mino = vc.index[1]

        ex_df = pd.concat([
            tmp[tmp[label_col] == maj].sample(
                n=min(3, vc.iloc[0]), random_state=42
            ),
            tmp[tmp[label_col] == mino].sample(
                n=min(2, vc.iloc[1]), random_state=42
            ),
        ], ignore_index=True)

        if len(ex_df) < 5:
            need = 5 - len(ex_df)
            ex_df = pd.concat(
                [ex_df, tmp.sample(n=need, random_state=123)],
                ignore_index=True
            )

        ex_df = ex_df.sample(frac=1, random_state=7).reset_index(drop=True)

    else:
        ex_df = tmp.sample(n=5, random_state=42).reset_index(drop=True)

    examples = [
        (row[text_col], row[label_col])
        for _, row in ex_df.iterrows()
    ]

    return examples

In [21]:
examples = select_five_examples(df, *find_columns(df))
pd.DataFrame(examples, columns=["Review", "Sentiment"])

,Review,Sentiment
0,"This is basically a goofball comedy, with some...",POSITIVE
1,"You can call this one a flop, and that's a ver...",NEGATIVE
2,"OK. Finally, a horror film that's done well. A...",POSITIVE
3,I went to see Antone Fisher not knowing what t...,POSITIVE
4,I saw this film in the movie theater. I was ta...,NEGATIVE


In [22]:
def format_examples(examples):
    blocks = []

    for i, (review, label) in enumerate(examples, 1):
        review = review.replace("\n", " ")  # clean newlines if needed

        block = (
            f"Example #{i}\n"
            f"Review: {review}\n"
            f"Sentiment: {label}\n"
        )

        blocks.append(block)

    return "\n".join(blocks)

In [23]:
few_shot_examples = format_examples(examples)
pprint(few_shot_examples)

('Example #1\n'
 'Review: This is basically a goofball comedy, with somewhat odd pacing due to '
 'some dramatic elements. For Michael J. Fox and Paul Reubens, it was their '
 'first film(Fox had previously been in a short lived TV series and a TV '
 'movie).<br /><br />Since the movie is basically a race/scavenger hunt type '
 'movie, like "Cannonball Run", "It\'s a Mad Mad Mad Mad World", or more '
 'recently "Rat Race", there are no main characters. Instead there are groups '
 'of characters, splitting the screen time and allowing for tons of '
 'mini-plots. Usually these kind of movies are a way to cram the maximum '
 'amount of stars (or semi-stars) onto a film.<br /><br />This one has teams, '
 'with one being the primary team which Fox belongs to, who are the only '
 'characters to be developed. Their plot is more of a stereotypical Disney '
 'affiar, about a college boy not paying attention to his younger brother, who '
 'he thinks of as a lazy punk. Since they are forced on a 

In [ ]:
few_shot_prompt = prompt + "\n\n" + few_shot_examples + "\n\n" + f"Now classify this new review:\n{review}\nLabel:"
pprint(few_shot_prompt)

('You are a sentiment classifier.\n'
 '\n'
 'Task:\n'
 'Classify the sentiment of the review as exactly one of:\n'
 'POSITIVE, NEGATIVE\n'
 '\n'
 'Rules:\n'
 '- Output ONLY the label (POSITIVE or NEGATIVE).\n'
 '- Do not add explanations.\n'
 '\n'
 'Review:\n'
 '{text}\n'
 '\n'
 'Label:\n'
 '\n'
 'Example #1\n'
 'Review: This is basically a goofball comedy, with somewhat odd pacing due to '
 'some dramatic elements. For Michael J. Fox and Paul Reubens, it was their '
 'first film(Fox had previously been in a short lived TV series and a TV '
 'movie).<br /><br />Since the movie is basically a race/scavenger hunt type '
 'movie, like "Cannonball Run", "It\'s a Mad Mad Mad Mad World", or more '
 'recently "Rat Race", there are no main characters. Instead there are groups '
 'of characters, splitting the screen time and allowing for tons of '
 'mini-plots. Usually these kind of movies are a way to cram the maximum '
 'amount of stars (or semi-stars) onto a film.<br /><br />This one has tea

In [26]:
client.generate(
    model="qwen3:0.6b",
    prompt=few_shot_prompt.format(text=review),
    think=False,
    stream=False,
).response

'POSITIVE'

In [27]:
label

'POSITIVE'

##### ICL (In-Context Learning)

In-Context Learning (ICL) is the broader concept behind few-shot prompting.

The model figures out what to do by reading examples inside your prompt — without changing its training.

* It does not update itself.
* It does not store memory.
* It just detects patterns in what you give it right now.

**Important**:
The model is not permanently learning — it’s temporarily adapting based on the examples you provide in that single prompt.

#### What is Fine Tuning?

Fine-tuning is when you take a pre-trained AI model and train it further on new, specific data so it performs better at a particular task.

Unlike ICL, fine-tuning actually changes the model.

It updates the model’s internal parameters (its weights).
That means the change is **permanent**, not temporary.

There are many fine tuning techniques, including:

* **LoRA (Low-Rank Adaptation)** is a technique for fine-tuning large language models without updating all their weights. Instead of retraining the entire model (which is huge and expensive), LoRA:

    * Freezes the original model weights

    * Adds small trainable matrices

    * Trains only those small additions
* **QLoRA**: Add a new idea **Quantization**. Normally model weights are stored in high precision (like 16-bit or 32-bit numbers). Therefore, QLoRA combine Quantization and LoRa adapters.


However... how can we fine tuning a Language Model?

##### Unsloth Library

[Unsloth](https://unsloth.ai/) is an open-source library that makes fine-tuning large language models faster and more memory-efficient, especially when using techniques like LoRA and QLoRA. It optimizes training so you can fine-tune models such as LLaMA or Mistral on limited hardware, like a single GPU, without needing massive computing resources. In short, Unsloth helps you train large models more efficiently and at lower cost by reducing memory usage and speeding up the fine-tuning process.

On their website, you can find many [tutorials](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide) on how to use it.

In the following example we will fine tuned a small language model using Unsloth using the IMDB dataset

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments, default_data_collator
from trl import SFTTrainer

We take 50 examples from the dataset

In [48]:
assert "review" in df.columns and "sentiment" in df.columns, df.columns
df = df.dropna(subset=["review", "sentiment"]).copy()
df["sentiment"] = df["sentiment"].astype(int)

n_total = 50
n_pos = min((df["sentiment"] == 1).sum(), n_total // 2)
n_neg = min((df["sentiment"] == 0).sum(), n_total - n_pos)

df_pos = df[df["sentiment"] == 1].sample(n=n_pos, random_state=42)
df_neg = df[df["sentiment"] == 0].sample(n=n_neg, random_state=42)
df_small = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Distribution:", df_small["sentiment"].value_counts().to_dict())


Distribution: {1: 25, 0: 25}


We build the prompts

In [ ]:
SYSTEM_PROMPT = (
    "You are a sentiment classification model."
    "Return ONLY one label: POSITIVE or NEGATIVE."
)

def label_to_text(y: int) -> str:
    return "POSITIVE" if int(y) == 1 else "NEGATIVE"

def build_messages(review: str, y: int):
    user = (
        "Classify the sentiment of the review as POSITIVE or NEGATIVE.\n\n"
        f"REVIEW:\n{review.strip()}\n\n"
        "Label:"
    )
    assistant = label_to_text(y)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]



We create a new ```messages``` column by formatting each review and its sentiment using ```build_messages```(), then convert that column into a Hugging Face Dataset object for training the model.

In [49]:
df_small["messages"] = df_small.apply(
    lambda r: build_messages(r["review"], r["sentiment"]), axis=1
)
ds = Dataset.from_pandas(df_small[["messages"]])
ds

Dataset({
    features: ['messages'],
    num_rows: 50
})

We will fine tuned ```Llama-3.2-1B-Instruct```, a 1-billion-parameter instruction-tuned language model designed to follow user prompts efficiently while being small enough to run on limited hardware

In [50]:
model_name = "unsloth/Llama-3.2-1B-Instruct"
max_seq_length = 512
load_in_4bit = True

We load the model and the tokenizer

In [51]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)


==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla V100-PCIE-32GB. Num GPUs = 8. Max memory: 31.733 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


To apply LoRA-based parameter-efficient fine-tuning (PEFT) to the base model by adding trainable low-rank adapters (with rank ```r=16```) to specific transformer layers (```q_proj```, ```k_proj```, ```v_proj```, ```o_proj```, ```gate_proj```, ```up_proj```, ```down_proj```), while keeping the original weights frozen, using optimized gradient checkpointing from Unsloth to reduce memory usage during training.

In [53]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


We first convert each example’s ```messages``` into a single formatted chat text using the tokenizer’s chat template, then we tokenize that text with real truncation and fixed-length padding, and finally we create the training labels by copying the ```input_ids``` (so the model learns to predict the next token) and remove the original columns to produce a fully tokenized dataset ready for training.

In [ ]:
def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds_text = ds.map(to_text)

def tokenize_fn(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",  
        return_tensors=None,
    )
    
    enc["labels"] = [ids.copy() for ids in enc["input_ids"]]
    return enc

ds_tok = ds_text.map(
    tokenize_fn,
    batched=True,
    remove_columns=ds_text.column_names,  
)

Map: 100%|██████████| 50/50 [00:00<00:00, 524.33 examples/s]


If you are using ```SFTTrainer``` and get an error like this:

```
AttributeError: 'int' object has no attribute 'mean'
```
It's because your loss function is returning an ```int``` instead of a PyTorch Tensor.

An easy, though not very elegant, way to solve this is to create a new class that extends ```SFTTrainer``` and handle this error in the ```compute_loss``` function.

In [10]:
class SafeSFTTrainer(SFTTrainer):
    """
    Custom SFTTrainer that safely computes loss by explicitly
    forwarding input_ids, attention_mask, and labels.

    This avoids subtle issues where labels may not be passed
    correctly by the default trainer.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Computes training loss using model's internal loss computation.

        Args:
            model: Language model.
            inputs: Batch dictionary containing tensors.
            return_outputs: Whether to also return model outputs.

        Returns:
            Loss tensor or (loss, outputs).
        """
        # Make sure labels exist (SFTTrainer’s collator usually provides them)
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            labels=inputs.get("labels", None),
        )
        loss = outputs.loss 
        return (loss, outputs) if return_outputs else loss

* We define the training setup using ```TrainingArguments```, specifying batch size, gradient accumulation (effective ```batch = 16```), learning rate, warmup, total training steps (60), logging/saving frequency, 8-bit AdamW optimizer, linear scheduler, and mixed precision (```fp16```) for efficiency.

* We then initialize ```SafeSFTTrainer``` by passing the model, tokenizer, tokenized dataset, training configuration, data collator, and maximum sequence length to prepare the model for supervised fine-tuning.

In [ ]:
out_dir = "imdb_unsloth_1b_llam3.2_instruct_lora"

training_args = TrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  
    learning_rate=2e-4,
    warmup_steps=5,
    max_steps=60,                   
    logging_steps=5,
    save_steps=60,
    optim="adamw_8bit",
    weight_decay=0.0,
    lr_scheduler_type="linear",
    fp16=True,          
    report_to="none",
)

trainer = SafeSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_tok,
    args=training_args,
    data_collator=default_data_collator,
    max_seq_length=max_seq_length,
)


We train the model

In [56]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 8 x 1) = 128
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
5,27.671800
10,21.588500
15,13.439000
20,9.856300
25,8.267200
30,8.062800
35,6.770000
40,7.034900
45,6.980900
50,6.552300


TrainOutput(global_step=60, training_loss=10.70804656346639, metrics={'train_runtime': 150.8288, 'train_samples_per_second': 50.919, 'train_steps_per_second': 0.398, 'total_flos': 9072399089664000.0, 'train_loss': 10.70804656346639, 'epoch': 60.0})

Finally, we store the model

In [ ]:
model.save_pretrained(out_dir)
tokenizer.save_pretrained(out_dir)
print("Training completed and saved to:", out_dir)


Training completed and saved to: imdb_unsloth_1b_lora


For use the fine tuned model during the inference phase, we need to upload and the compute the evalaution metrics

In [57]:
FastLanguageModel.for_inference(model)

def predict_sentiment(review: str):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Classify the sentiment of the review as POSITIVE or NEGATIVE.\n\n"
            f"REVIEW:\n{review.strip()}\n\n"
            "Label:"
        )},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        temperature=0.0,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True).upper()
    if "POSITIVE" in decoded:
        return "POSITIVE"
    if "NEGATIVE" in decoded:
        return "NEGATIVE"
    return decoded[-50:]

In [58]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

df_eval = df.sample(n=15, random_state=42).reset_index(drop=True)
y_true = df_eval["sentiment"].astype(int).tolist()
y_pred = [1 if predict_sentiment(r) == "POSITIVE" else 0 for r in df_eval["review"].tolist()]

print({
    "accuracy": accuracy_score(y_true, y_pred),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    "f1_macro": f1_score(y_true, y_pred, average="macro"),
    "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
})

{'accuracy': 0.5333333333333333, 'f1_weighted': 0.3710144927536232, 'f1_macro': 0.34782608695652173, 'precision': 0.26666666666666666, 'recall': 0.5}
